In [1]:
import sys
import os

# For notebooks, use getcwd(); for scripts, use __file__
try:
    notebook_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    # Running in a notebook
    notebook_dir = os.getcwd()

sys.path.insert(0, notebook_dir)

from agents.orchestrator import analyze_user_input, update_state_from_analysis
from agents.specialist import run_activities_specialist, run_logistics_specialist
from agents.verifier import verify_and_format_itinerary
from state import TravelState


In [2]:
state = TravelState()
task_json = {
  "task_id": "task_123",
  "intent": "update_constraints",
  "updated_constraints": {
    "origin": "BOS",
    "destination": "SEA",
    "start_date": "2026-05-05",
    "end_date": "2026-05-12"
  },
  "delegation": "logistics",
  "response_to_user": "I will search for flights...",
}
user_input = None #"want to change my trip to Seattle from May 6th to May 13th"

analysis = analyze_user_input(user_input, state, task_json=task_json)
state = update_state_from_analysis(state, analysis)
delegation = analysis.get("delegation", "none")

In [3]:
 # 3. Delegation
if delegation == "none":
    response = analysis.get(
        "response_to_user",
        "I'm still missing some info to start planning.",
    )
    print(f"\nNomad: {response}")
    state.messages.append({"role": "assistant", "content": response})

In [4]:

# If we reach here, we need to run specialists
draft_components = []
constraints_str = state.constraints.model_dump_json(indent=2)

if delegation in ["logistics", "both"]:
    print("\n[Specialist - Logistics] Searching for flights & hotels...")
    try:
        logistics_draft = run_logistics_specialist(constraints_str, task_id=state.task_id)
        draft_components.append("--- LOGISTICS ---\n" + logistics_draft)
    except Exception as e:
        print(f"[Specialist Error] {e}")


[Specialist - Logistics] Searching for flights & hotels...
  [Specialist Turn 1] Thinking...
  🔍 LLM response received with stop reason: tool_use
  🧩 Content blocks: [{'type': 'text', 'text': "I'll search for flights and hotels for your trip from Boston to Seattle from May 5-12, 2026. Let me find the best options for you."}, {'type': 'tool_use', 'id': 'toolu_01XPezDhU6t39NpRLcBHoFHW', 'name': 'search_flights', 'input': {'origin': 'BOS', 'destination': 'SEA', 'departure_date': '2026-05-05', 'return_date': '2026-05-12'}, 'caller': {'type': 'direct'}}, {'type': 'tool_use', 'id': 'toolu_01MXncXj86kZNRVef3xBzaPr', 'name': 'search_hotels', 'input': {'location': 'Seattle, Washington', 'check_in': '2026-05-05', 'check_out': '2026-05-12', 'adults': 1}, 'caller': {'type': 'direct'}}]
  🔧 Executing tool search_flights with arguments {'origin': 'BOS', 'destination': 'SEA', 'departure_date': '2026-05-05', 'return_date': '2026-05-12', 'task_id': 'task_123', 'turn': 0}
[Cache Hit] google_flights
🍽️ 

In [5]:
# 4. Verifier: check draft against constraints
print("\n[Verifier] Checking itinerary against constraints...")
full_draft = "\n\n".join(draft_components)

try:
    verification = verify_and_format_itinerary(full_draft, constraints_str)

    if verification.get("is_valid"):
        final_response = verification.get(
            "final_message_to_user", "Here is your plan!"
        )
        print("\nNomad:\n")
        print(final_response)
        state.messages.append({"role": "assistant", "content": final_response})
    else:
        issues = verification.get("issues", [])
        print(
            f"\nNomad: I built a plan, but it violates some constraints:\n{issues}"
        )
        print("I will need to revise. Let me know how you'd like to adjust.")

except Exception as e:
    print(f"Error during verification: {e}")


[Verifier] Checking itinerary against constraints...

Nomad:

# ✅ Your Boston to Seattle Itinerary (May 5-12, 2026)

Your itinerary has been **verified and approved**! All options meet your travel requirements.

---

## ✈️ **FLIGHT OPTIONS**

### **Best Value: Southwest Airlines - $210 Round Trip**
- **Outbound (May 5):** BOS → MDW → SEA
  - Depart: 6:20 PM from Boston Logan
  - Arrive: 12:10 AM (May 6) in Seattle
  - Layover: 90 min in Chicago Midway
  - Total duration: 8h 50min
  - **Carbon emissions: 10% lower than typical**
  - Free Wi-Fi, checked baggage for a fee

### **Fastest & Most Convenient: Delta Airlines - $297 Round Trip**
- **Outbound (May 5):** BOS → SEA (Nonstop)
  - Depart: 7:37 AM from Boston Logan
  - Arrive: 10:55 AM in Seattle
  - Duration: 6h 18min
  - Aircraft: Airbus A321neo
  - **Carbon emissions: 27% lower than typical**
  - Free Wi-Fi, in-seat power & USB outlets, Live TV
  - Alternative departure at 11:10 AM also available

### **Other Options:**
- **Unite

In [ ]:

def main():
    print("Welcome to Nomad Travel Agent!")
    print("Type 'quit' or 'exit' to stop.")
    print("-----------------------------------")

    state = TravelState()

    while True:
        user_input = input("\nYou: ")
        if user_input.lower() in ["quit", "exit"]:
            break

        # 1. Orchestrator: classify intent, extract constraints, decide delegation
        print("\n[Orchestrator] Analyzing input and updating Constraint Layer...")
        try:
            analysis = analyze_user_input(user_input, state)
            state = update_state_from_analysis(state, analysis)

            print(f"[Orchestrator] Intent: {analysis.get('intent')}")
            print(f"[Orchestrator] Delegation: {analysis.get('delegation')}")

            delegation = analysis.get("delegation", "none")

        except Exception as e:
            print(f"Error during orchestration: {e}")
            continue

        # 2. Add to message history
        state.messages.append({"role": "user", "content": user_input})

        # 3. Delegation
        if delegation == "none":
            response = analysis.get(
                "response_to_user",
                "I'm still missing some info to start planning.",
            )
            print(f"\nNomad: {response}")
            state.messages.append({"role": "assistant", "content": response})
            continue

        # If we reach here, we need to run specialists
        draft_components = []
        constraints_str = state.constraints.model_dump_json(indent=2)

        if delegation in ["logistics", "both"]:
            print("\n[Specialist - Logistics] Searching for flights & hotels...")
            try:
                logistics_draft = run_logistics_specialist(constraints_str)
                draft_components.append("--- LOGISTICS ---\n" + logistics_draft)
            except Exception as e:
                print(f"[Specialist Error] {e}")

        if delegation in ["activities", "both"]:
            print(
                "\n[Specialist - Activities] Searching for restaurants & things to do..."
            )
            try:
                activities_draft = run_activities_specialist(constraints_str)
                draft_components.append("--- ACTIVITIES ---\n" + activities_draft)
            except Exception as e:
                print(f"[Specialist Error] {e}")

        # 4. Verifier: check draft against constraints
        print("\n[Verifier] Checking itinerary against constraints...")
        full_draft = "\n\n".join(draft_components)

        try:
            verification = verify_and_format_itinerary(full_draft, constraints_str)

            if verification.get("is_valid"):
                final_response = verification.get(
                    "final_message_to_user", "Here is your plan!"
                )
                print("\nNomad:\n")
                print(final_response)
                state.messages.append({"role": "assistant", "content": final_response})
            else:
                issues = verification.get("issues", [])
                print(
                    f"\nNomad: I built a plan, but it violates some constraints:\n{issues}"
                )
                print("I will need to revise. Let me know how you'd like to adjust.")

        except Exception as e:
            print(f"Error during verification: {e}")


if __name__ == "__main__":
    main()